## חיבור דמוגרפיה, קהל ותקציבים

במחברת הזו:
- טוענים את טבלת הדמוגרפיה לפי עונה
- מחברים אליה נתוני קהל לפי קבוצה ועונה
- מחברים אליה נתוני תקציב לפי קבוצה ועונה
- שומרים קבצי CSV מוכנים לניתוח המשך

הערה: נתוני התקציב אינם מלאים לכל הקבוצות בכל העונות, ולכן יישארו ערכי חסר בחלק מהשורות.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'notebooks' / 'data').exists():
            return candidate
    return current


ROOT = find_project_root()
DATA_DIR = ROOT / 'notebooks' / 'data'
DEMOGRAPHIC_DIR = DATA_DIR / 'demographic'
ATTENDANCE_DIR = DATA_DIR / 'attendance'
ECONOMIC_DIR = DATA_DIR / 'economic_data'

demographic_df = pd.read_csv(DEMOGRAPHIC_DIR / 'ligat_haal_teams_city_population_by_season.csv')
attendance_df = pd.read_csv(ATTENDANCE_DIR / 'attendance_all_seasons_ligat_haal_transfermarkt.csv')
budget_df = pd.read_csv(ECONOMIC_DIR / 'budget_data_combined_all_sources.csv')

attendance_name_map = {
    'Ihud Bnei Sakhnin': 'Bnei Sakhnin',
    'Hapoel Ramat haSharon': 'Hapoel Nir Ramat HaSharon',
    'Hapoel Rishon leZion': 'Hapoel Rishon LeZion',
    'Sekzia Ness Ziona': 'Sektzia Ness Ziona',
}

budget_name_map = {
    'Bnei Reineh': 'Maccabi Bnei Reineh',
    "Hapoel Be'er Sheva": 'Hapoel Beer Sheva',
    'Hapoel Kiryat Shmona': 'Ironi Kiryat Shmona',
    'Ironi Kiryat Shmona': 'Ironi Kiryat Shmona',
    'MS Ashdod': 'FC Ashdod',
}

attendance_prepared = attendance_df.rename(columns={'team': 'team_attendance_raw'}).copy()
attendance_prepared['club_name'] = attendance_prepared['team_attendance_raw'].replace(attendance_name_map)
for column in ['capacity', 'total_spectators', 'average_attendance']:
    attendance_prepared[column] = pd.to_numeric(attendance_prepared[column], errors='coerce')

attendance_joined_df = demographic_df.merge(
    attendance_prepared[
        ['season', 'club_name', 'team_attendance_raw', 'stadium', 'capacity', 'total_spectators', 'average_attendance']
    ],
    on=['season', 'club_name'],
    how='left',
)
attendance_joined_df['attendance_per_1000_residents'] = (
    attendance_joined_df['average_attendance'] / attendance_joined_df['population_total'] * 1000
    )
attendance_joined_df['total_spectators_per_resident'] = (
    attendance_joined_df['total_spectators'] / attendance_joined_df['population_total']
    )
attendance_joined_df['average_attendance_pct_of_population'] = (
    attendance_joined_df['average_attendance'] / attendance_joined_df['population_total'] * 100
    )
attendance_joined_df['stadium_capacity_utilization_pct'] = (
    attendance_joined_df['average_attendance'] / attendance_joined_df['capacity'] * 100
    )

budget_prepared = budget_df.rename(columns={'team_english': 'team_budget_raw'}).copy()
budget_prepared = budget_prepared[~budget_prepared['team_budget_raw'].isin(['Rest of league', 'Kiryat Shmona / Hapoel Be\'er Sheva'])].copy()
budget_prepared['club_name'] = budget_prepared['team_budget_raw'].replace(budget_name_map)
budget_prepared['budget_source_priority'] = np.where(
    budget_prepared['data_source'].astype(str).str.contains('Gemini', case=False, na=False),
    1,
    0,
    )
budget_prepared = (
    budget_prepared
    .sort_values(['season', 'club_name', 'budget_source_priority', 'budget_mid_million_nis'], ascending=[True, True, True, False])
    .drop_duplicates(['season', 'club_name'], keep='first')
    .reset_index(drop=True)
)
budget_prepared['budget_mid_nis'] = budget_prepared['budget_mid_million_nis'] * 1_000_000
budget_prepared['budget_min_nis'] = budget_prepared['budget_min_million_nis'] * 1_000_000
budget_prepared['budget_max_nis'] = budget_prepared['budget_max_million_nis'] * 1_000_000

budget_joined_df = demographic_df.merge(
    budget_prepared[
        [
            'season',
            'club_name',
            'team_budget_raw',
            'budget_min_million_nis',
            'budget_max_million_nis',
            'budget_mid_million_nis',
            'budget_min_nis',
            'budget_max_nis',
            'budget_mid_nis',
            'data_source',
            'source_note',
        ]
    ],
    on=['season', 'club_name'],
    how='left',
)
budget_joined_df['budget_nis_per_resident'] = (
    budget_joined_df['budget_mid_nis'] / budget_joined_df['population_total']
    )
budget_joined_df['budget_million_nis_per_100k_residents'] = (
    budget_joined_df['budget_mid_million_nis'] / budget_joined_df['population_total'] * 100000
    )

combined_joined_df = attendance_joined_df.merge(
    budget_joined_df[
        [
            'season',
            'club_name',
            'team_budget_raw',
            'budget_min_million_nis',
            'budget_max_million_nis',
            'budget_mid_million_nis',
            'budget_min_nis',
            'budget_max_nis',
            'budget_mid_nis',
            'data_source',
            'source_note',
            'budget_nis_per_resident',
            'budget_million_nis_per_100k_residents',
        ]
    ],
    on=['season', 'club_name'],
    how='left',
)
combined_joined_df['attendance_per_budget_million'] = (
    combined_joined_df['average_attendance'] / combined_joined_df['budget_mid_million_nis']
    )
combined_joined_df['total_spectators_per_budget_million'] = (
    combined_joined_df['total_spectators'] / combined_joined_df['budget_mid_million_nis']
    )

coverage_summary_df = (
    combined_joined_df.groupby(['season', 'season_year'], as_index=False)
    .agg(
        teams=('club_name', 'count'),
        attendance_matches=('average_attendance', lambda s: int(s.notna().sum())),
        budget_matches=('budget_mid_million_nis', lambda s: int(s.notna().sum())),
        matches_with_both=('attendance_per_budget_million', lambda s: int(s.notna().sum())),
        total_population=('population_total', 'sum'),
        total_spectators=('total_spectators', 'sum'),
        total_budget_million_nis=('budget_mid_million_nis', 'sum'),
    )
    .sort_values('season_year')
    .reset_index(drop=True)
)
coverage_summary_df['attendance_match_rate_pct'] = coverage_summary_df['attendance_matches'] / coverage_summary_df['teams'] * 100
coverage_summary_df['budget_match_rate_pct'] = coverage_summary_df['budget_matches'] / coverage_summary_df['teams'] * 100
coverage_summary_df['both_match_rate_pct'] = coverage_summary_df['matches_with_both'] / coverage_summary_df['teams'] * 100

attendance_output = DEMOGRAPHIC_DIR / 'ligat_haal_demographics_attendance_by_season.csv'
budget_output = DEMOGRAPHIC_DIR / 'ligat_haal_demographics_budget_by_season.csv'
combined_output = DEMOGRAPHIC_DIR / 'ligat_haal_demographics_attendance_budget_by_season.csv'
coverage_output = DEMOGRAPHIC_DIR / 'ligat_haal_demographics_join_coverage_by_season.csv'

attendance_joined_df.to_csv(attendance_output, index=False, encoding='utf-8-sig')
budget_joined_df.to_csv(budget_output, index=False, encoding='utf-8-sig')
combined_joined_df.to_csv(combined_output, index=False, encoding='utf-8-sig')
coverage_summary_df.to_csv(coverage_output, index=False, encoding='utf-8-sig')

print(f'Demographic rows: {len(demographic_df):,}')
print(f'Attendance join matches: {attendance_joined_df["average_attendance"].notna().sum():,}')
print(f'Budget join matches: {budget_joined_df["budget_mid_million_nis"].notna().sum():,}')
print(f'Combined rows with both: {combined_joined_df["attendance_per_budget_million"].notna().sum():,}')
print('Saved files:')
print(attendance_output)
print(budget_output)
print(combined_output)
print(coverage_output)

Demographic rows: 280
Attendance join matches: 280
Budget join matches: 120
Combined rows with both: 120
Saved files:
C:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\demographic\ligat_haal_demographics_attendance_by_season.csv
C:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\demographic\ligat_haal_demographics_budget_by_season.csv
C:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\demographic\ligat_haal_demographics_attendance_budget_by_season.csv
C:\Users\nitib\dev-lab\ligat_haal_project\ligat_haal_project\notebooks\data\demographic\ligat_haal_demographics_join_coverage_by_season.csv


In [2]:
display(coverage_summary_df)

display(
    combined_joined_df[
        [
            'season',
            'club_name',
            'city_he',
            'population_total',
            'average_attendance',
            'total_spectators',
            'budget_mid_million_nis',
            'attendance_per_1000_residents',
            'budget_nis_per_resident',
            'attendance_per_budget_million',
        ]
    ].head(30)
 )

,season,season_year,teams,attendance_matches,budget_matches,matches_with_both,total_population,total_spectators,total_budget_million_nis,attendance_match_rate_pct,budget_match_rate_pct,both_match_rate_pct
0,2006/07,2006,12,12,4,4,4218651.0,119700,257.5,100.0,33.333333,33.333333
1,2007/08,2007,12,12,4,4,3834169.0,362600,257.5,100.0,33.333333,33.333333
2,2008/09,2008,12,12,4,4,4066519.0,0,232.5,100.0,33.333333,33.333333
3,2009/10,2009,16,16,4,4,4774303.0,939155,210.0,100.0,25.000000,25.000000
4,2010/11,2010,16,16,4,4,4806938.0,318450,247.5,100.0,25.000000,25.000000
5,2011/12,2011,16,16,4,4,4775966.0,911780,230.5,100.0,25.000000,25.000000
6,2012/13,2012,14,14,4,4,4148127.0,916940,247.5,100.0,28.571429,28.571429
7,2013/14,2013,14,14,4,4,4094439.0,970781,257.5,100.0,28.571429,28.571429
8,2014/15,2014,14,14,5,5,4055894.0,935937,315.5,100.0,35.714286,35.714286
9,2015/16,2015,14,14,5,5,4151239.0,1247497,351.0,100.0,35.714286,35.714286


,season,club_name,city_he,population_total,average_attendance,total_spectators,budget_mid_million_nis,attendance_per_1000_residents,budget_nis_per_resident,attendance_per_budget_million
0,2006/07,FC Ashdod,אשדוד,228562.0,2500,2500,NaN,10.937951,NaN,NaN
1,2006/07,Maccabi Herzliya,הרצלייה,110885.0,1000,1000,NaN,9.018352,NaN,NaN
2,2006/07,Maccabi Haifa,חיפה,297083.0,3850,7700,50.0,12.959341,168.303134,77.000000
3,2006/07,Beitar Jerusalem,ירושלים,1050151.0,10000,10000,135.0,9.522440,128.552941,74.074074
4,2006/07,Hapoel Kfar Saba,כפר סבא,99410.0,2250,4500,NaN,22.633538,NaN,NaN
5,2006/07,Maccabi Netanya,נתניה,234812.0,3083,9250,NaN,13.129653,NaN,NaN
6,2006/07,Hapoel Petah Tikva,פתח תקווה,270403.0,2050,10250,NaN,7.581277,NaN,NaN
7,2006/07,Maccabi Petah Tikva,פתח תקווה,270403.0,2000,2000,NaN,7.396368,NaN,NaN
8,2006/07,Hakoah Amidar Ramat Gan,רמת גן,172242.0,1250,6250,NaN,7.257231,NaN,NaN
9,2006/07,Bnei Yehuda Tel Aviv,תל אביב -יפו,494900.0,3063,49000,NaN,6.189129,NaN,NaN
